In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam

# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------
DATASET_PATH = "/content/drive/MyDrive/Datasets/sugarcane"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Datasets/sugarcane/model_sugarcane_disease.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 12  # slightly higher because dataset is small


# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------
def load_dataset(path):
    filepaths = []
    labels = []

    for class_name in os.listdir(path):
        class_path = os.path.join(path, class_name)

        if os.path.isdir(class_path):
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".png", ".jpeg")):
                    filepaths.append(os.path.join(class_path, img))
                    labels.append(class_name)

    return pd.DataFrame({"Filepath": filepaths, "Label": labels})


# ---------------------------------------------------------
# Hybrid Model
# ---------------------------------------------------------
def build_model(num_classes):

    effnet = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    effnet.trainable = False

    convnext = tf.keras.applications.ConvNeXtTiny(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(224, 224, 3)
    )
    convnext.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))

    x1 = effnet(inputs)
    x2 = convnext(inputs)

    x = tf.keras.layers.Concatenate()([x1, x2])
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)

    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)


# ---------------------------------------------------------
# Train
# ---------------------------------------------------------
def train():

    print("GPU Available:", tf.config.list_physical_devices('GPU'))

    print("Loading dataset...")
    df = load_dataset(DATASET_PATH)

    # Manual 80/20 split
    train_df, val_df = train_test_split(
        df,
        train_size=0.8,
        stratify=df["Label"],
        random_state=42
    )

    # 🔥 Strong augmentation (important for small dataset)
    train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
        rotation_range=40,
        zoom_range=0.4,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.3,
        brightness_range=[0.5, 1.5],
        horizontal_flip=True,
        fill_mode="nearest"
    )

    val_gen = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
    )

    train_images = train_gen.flow_from_dataframe(
        train_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    val_images = val_gen.flow_from_dataframe(
        val_df,
        x_col="Filepath",
        y_col="Label",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical"
    )

    print("Classes:", train_images.class_indices)

    model = build_model(len(train_images.class_indices))

    model.compile(
        optimizer=Adam(1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("Training on GPU 🚀 ...")
    history = model.fit(
        train_images,
        validation_data=val_images,
        epochs=EPOCHS
    )

    print("Saving model...")
    model.save(MODEL_SAVE_PATH)

    # Save plots
    plt.figure()
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.legend(["Train", "Val"])
    plt.title("Accuracy")
    plt.savefig("/content/drive/MyDrive/Datasets/sugarcane/accuracy.png")
    plt.close()

    plt.figure()
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.legend(["Train", "Val"])
    plt.title("Loss")
    plt.savefig("/content/drive/MyDrive/Datasets/sugarcane/loss.png")
    plt.close()


train()

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Loading dataset...
Found 2016 validated image filenames belonging to 5 classes.
Found 505 validated image filenames belonging to 5 classes.
Classes: {'Healthy': 0, 'Mosaic': 1, 'RedRot': 2, 'Rust': 3, 'Yellow': 4}
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training on GPU 🚀 ...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/12
63/63 ━━━━━━━━━━━━━━━━━━━━ 693s 10s/step - accuracy: 0.2609 - loss: 1.7661 - val_accuracy: 0.4832 - val_loss: 1.2517
Epoch 2/12
63/63 ━━━━━━━━━━━━━━━━━━━━ 54s 861ms/step - accuracy: 0.4667 - loss: 1.2703 - val_accuracy: 0.6040 - val_loss: 1.0337
Epoch 3/12
63/63 ━━━━━━━━━━━━━━━━━━━━ 54s 851ms/step - accuracy: 0.5983 - loss: 1.0514 - val_accuracy: 0.6614 - val_loss: 0.8730
Epoch 4/12
63/63 ━━━━━━━━━━━━━━━━━━━━ 53s 842ms/step - accuracy: 0.6536 - loss: 0.9251 - val_accuracy: 0.7050 - val_loss: 0.7629
Epoch 5/12
63/63 ━━━━━━━━━━━━━━━━━━━━ 54s 851ms/step - accuracy: 0.6705 - loss: 0.8405 - val_accuracy: 0.7545 - val_loss: 0.7037
Epoch 6/12
63/63 ━━━━━━━━━━━━━━━━━━━━ 54s 865ms/step - accuracy: 0.6946 - loss: 0.7822 - val_accuracy: 0.7545 - val_loss: 0.6615
Epoch 7/12
63/63 ━━━━━━━━━━━━━━━━━━━━ 54s 861ms/step - accuracy: 0.7473 - loss: 0.7051 - val_accuracy: 0.7901 - val_loss: 0.5847
Epoch 8/12
63/63 ━━━━━━━━━━━━━━━━━━━━ 53s 841ms/step - accuracy: 0.7687 - loss: 0.6460 - val_accur